<a href="https://colab.research.google.com/github/FrankChen0930/MarketMamba/blob/main/MarketMamba_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 🚀 MarketMamba V2: Cell 1 - 資料鍛造廠 (終極防爆版)
# 核心：絕對時間軸對齊 + 儲存輕量化基礎矩陣 (避免 64GB OOM)
# ==========================================
import os
import torch
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm.auto import tqdm
from google.colab import drive

print("🏗️ 啟動 MarketMamba V2 資料鍛造廠 (終極防爆版)...\n")

# ==========================================
# 1. 掛載與路徑設定
# ==========================================
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/MarketMamba_Database'
RAW_DATA_DIR = os.path.join(BASE_DIR, 'Macro_1D_History')
FEATURE_DIR = os.path.join(BASE_DIR, 'Macro_1D_Features')

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

# ==========================================
# 2. 獲取宏觀大盤與匯率
# ==========================================
print("🌍 正在連線 Yahoo Finance 獲取總體經濟特徵...")
taiex = yf.download("^TWII", period="max", progress=False)
taiex['Market_Return'] = taiex['Close'].pct_change()

usdtwd = yf.download("USDTWD=X", period="max", progress=False)
usdtwd['USD_TWD'] = usdtwd['Close']

macro_df = pd.DataFrame(index=taiex.index)
macro_df = macro_df.join(taiex['Market_Return']).join(usdtwd['USD_TWD']).ffill().bfill()
macro_df.index = pd.to_datetime(macro_df.index).tz_localize(None)

MACRO_SAVE_PATH = os.path.join(RAW_DATA_DIR, 'TAIEX_Macro.parquet')
macro_df.to_parquet(MACRO_SAVE_PATH)

# ==========================================
# 3. 特徵提煉與絕對時間軸對齊
# ==========================================
files = [f for f in os.listdir(RAW_DATA_DIR) if f.endswith('.parquet') and f != 'TAIEX_Macro.parquet']
all_stocks_data = []
valid_tickers = []

# 🌟 提取大盤的時間軸，作為全市場的「絕對標準」
master_dates = macro_df.index
print(f"\n📏 確立全市場絕對時間軸：共 {len(master_dates)} 個交易日")
print(f"🧪 開始為 {len(files)} 檔股票注入技術指標並強制對齊時間軸...")

for f in tqdm(files, desc="特徵提煉中"):
    ticker = f.split('_')[0]
    df = pd.read_parquet(os.path.join(RAW_DATA_DIR, f))
    df.index = pd.to_datetime(df.index).tz_localize(None)

    # 融合宏觀特徵
    df = df.join(macro_df, how='inner')

    # 計算進階特徵
    df['MA20'] = df['Close'].rolling(window=20).mean()
    df['MA20_Bias'] = (df['Close'] - df['MA20']) / (df['MA20'] + 1e-8)

    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-8)
    df['RSI_14'] = 100 - (100 / (1 + rs))

    df['Daily_Return'] = df['Close'].pct_change()
    df['Vol_10D'] = df['Daily_Return'].rolling(window=10).std()

    df = df.bfill().ffill()

    feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume',
                    'Market_Return', 'USD_TWD', 'MA20_Bias', 'RSI_14', 'Vol_10D']

    # 標準化 (Z-score)
    for col in feature_cols:
        df[col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-8)

    # 🌟 強制對齊並填補空洞
    df_aligned = df[feature_cols].reindex(master_dates)
    df_aligned = df_aligned.ffill().bfill().fillna(0)

    all_stocks_data.append(df_aligned.values)
    valid_tickers.append(ticker)

# ==========================================
# 4. 終極防爆存檔 (只存基礎矩陣，不切片！)
# ==========================================
market_array = np.stack(all_stocks_data, axis=1)
print(f"\n✅ 全市場特徵矩陣構建完成！形狀: {market_array.shape}")

# 轉成 Float32 節省記憶體
market_tensor = torch.tensor(market_array, dtype=torch.float32)
tensor_size_mb = market_tensor.element_size() * market_tensor.nelement() / (1024 * 1024)
print(f"📦 基礎時空矩陣大小: 僅 {tensor_size_mb:.2f} MB (完美避開 OOM 記憶體炸彈！)")

# 安全歸檔至 Macro_1D_Features
TENSOR_SAVE_PATH = os.path.join(FEATURE_DIR, 'market_tensors_v2_base.pt')
torch.save({
    'market_matrix': market_tensor,
    'tickers': valid_tickers,
    'features': feature_cols
}, TENSOR_SAVE_PATH)

print(f"🎉 偉大的 V2 基礎時空矩陣已完美歸檔至: \n 📂 {TENSOR_SAVE_PATH}")

In [ ]:
# ==========================================
# 🩺 MarketMamba V2: Cell 1.5 - 資料庫 X 光體檢中心
# 功能：驗證 V2 基礎矩陣的形狀、大小、以及是否有致命的 NaN/Inf
# ==========================================
import os
import torch
import numpy as np

print("🕵️‍♂️ 啟動 MarketMamba V2 資料庫 X光體檢中心...\n")

FEATURE_DIR = '/content/drive/MyDrive/MarketMamba_Database/Macro_1D_Features'
TENSOR_SAVE_PATH = os.path.join(FEATURE_DIR, 'market_tensors_v2_base.pt')

# 1. 檢查檔案是否存在
if not os.path.exists(TENSOR_SAVE_PATH):
    print(f"❌ 找不到檔案！請確認路徑: {TENSOR_SAVE_PATH}")
else:
    # 2. 嘗試載入資料
    print(f"📦 正在載入時空矩陣，請稍候...")
    try:
        data = torch.load(TENSOR_SAVE_PATH, weights_only=False)
        matrix = data.get('market_matrix')
        tickers = data.get('tickers')
        features = data.get('features')

        print("\n📊 【基礎資料結構檢查】")
        print(f" ┣ 📏 矩陣形狀: {matrix.shape} (預期: 絕對天數, 股票數, 10特徵)")
        print(f" ┣ 🏢 股票數量: {len(tickers)} 檔")
        print(f" ┣ 🧬 特徵數量: {len(features)} 個")
        print(f" ┣ 📝 特徵清單: {', '.join(features)}")
        print(f" ┗ 💾 資料型態: {matrix.dtype}")

        # 3. 檢查實際佔用的記憶體大小
        tensor_size_mb = matrix.element_size() * matrix.nelement() / (1024 * 1024)
        print(f"\n⚖️ 記憶體負載評估: ~{tensor_size_mb:.2f} MB")
        if tensor_size_mb < 2000:
            print(" ✅ 容量極度健康，完全不會撐爆 RAM！")
        else:
            print(" ⚠️ 容量偏大，請小心記憶體限制。")

        # 4. 致命毒藥檢測 (NaN & Inf)
        # 神經網路只要吃到一個 NaN，Loss 就會直接變成 NaN，整個模型就毀了
        print("\n🧪 【致命毒藥 (NaN/Inf) 深度檢測】")
        has_nan = torch.isnan(matrix).any().item()
        has_inf = torch.isinf(matrix).any().item()

        print(f" ┣ NaN (缺失值) 檢測: {'❌ 發現致命 NaN!' if has_nan else '✅ 完美無瑕，無缺失值'}")
        print(f" ┗ Inf (無限大) 檢測: {'❌ 發現致命 Inf!' if has_inf else '✅ 完美無瑕，無無限大數值'}")

        # 5. 隨機抽查數值分佈 (以第一檔股票的第一個特徵為例)
        print(f"\n🔍 【數值分佈抽查 (以 {tickers[0]} 的 {features[0]} 特徵為例)】")
        test_stock_feature = matrix[:, 0, 0]
        # 排除掉因為還沒上市而填補的 0
        non_zero_elements = test_stock_feature[test_stock_feature != 0]

        if len(non_zero_elements) > 0:
            mean_val = non_zero_elements.mean().item()
            std_val = non_zero_elements.std().item()
            print(f" ┣ 活躍交易天數: {len(non_zero_elements)} 天")
            print(f" ┣ 平均值 (Mean): {mean_val:.4f} (Z-score 標準化後應接近 0)")
            print(f" ┗ 標準差 (Std):  {std_val:.4f} (Z-score 標準化後應接近 1)")

        print("\n==========================================")
        if not has_nan and not has_inf:
            print("🎉 體檢報告：SSS 級完美過關！")
            print("🚀 你的 V2 資料庫已經堅不可摧，隨時可以關閉 CPU 機器，換 A100 準備煉丹！")
        else:
            print("⚠️ 體檢報告：發現異常，請勿直接進入訓練！")
        print("==========================================")

    except Exception as e:
        print(f"❌ 檔案讀取失敗，可能是存檔過程中損壞。錯誤訊息: {e}")

In [ ]:
import os
from google.colab import drive

print("🔌 正在掛載 Google Drive...")
drive.mount('/content/drive')

print("\n🚀 啟動 A100 極速武裝程序...")

# 1. 安裝圖神經網路核心套件 (PyTorch Geometric)
print("🕸️ 安裝 torch-geometric...")
!pip install -q torch-geometric

# 2. 尋找並安裝你編譯好的 Mamba 專武 (.whl)
# 假設你的 wheel 檔存在這個資料夾，如果路徑不同請幫我微調一下
WHEEL_DIR = '/content/drive/MyDrive/Mamba_Wheels_A100'

print(f"📦 從 {WHEEL_DIR} 載入預編譯套件...")
!pip install -q {WHEEL_DIR}/causal_conv1d-*.whl
!pip install -q {WHEEL_DIR}/mamba_ssm-*.whl

print("\n==========================================")
print("✅ Cell 1 執行完畢！")
print("A100 環境已完美配置 Mamba 與 GNN 套件！")
print("==========================================")

In [ ]:
# ==========================================
# 🚀 MarketMamba V3: Cell 2 - 擴容版神經網路大腦
# 核心升級：d_model 翻倍 (128) + 預測未來 20 天連續軌跡
# ==========================================
import torch
import torch.nn as nn
import math
from mamba_ssm import Mamba
from torch_geometric.nn import GATConv

print("🧠 正在建構 MarketMamba V3 擴容版大腦...")

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class MarketMambaV3(nn.Module):
    # ⚠️ 升級 1：d_model 從 64 翻倍到 128，GAT 注意力頭數增加到 8
    def __init__(self, input_dim=10, d_model=128, num_heads=8, pred_days=30):
        super().__init__()
        self.d_model = d_model
        self.pred_days = pred_days

        self.feature_proj = nn.Linear(input_dim, d_model)

        # ⚠️ 升級 2：Mamba 引擎擴容
        self.mamba = Mamba(
            d_model=d_model,
            d_state=32,  # 狀態空間翻倍，記住更多 120 天的細節
            d_conv=4,
            expand=2,
        )

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        # 因為現在目標是 20 天的軌跡，所以雜訊輸入也是 20 維
        self.noise_proj = nn.Linear(pred_days, d_model)

        self.gat = GATConv(d_model, d_model // num_heads, heads=num_heads, concat=True)

        # ⚠️ 升級 3：輸出層從吐出 1 個數字，變成吐出 20 天的連續軌跡
        self.output_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, pred_days) # 最後吐出 20 個數字
        )

    def forward(self, x_history, edge_index, noisy_target, time_step):
        batch_size, seq_len, num_stocks, _ = x_history.shape

        x_mamba = x_history.permute(0, 2, 1, 3).reshape(batch_size * num_stocks, seq_len, -1)
        x_mamba = self.feature_proj(x_mamba)
        x_mamba = self.mamba(x_mamba)

        x_mamba_out = x_mamba[:, -1, :].reshape(batch_size, num_stocks, self.d_model)

        t_emb = self.time_mlp(time_step).unsqueeze(1).expand(-1, num_stocks, -1)

        # 處理 20 維的軌跡雜訊
        noise_emb = self.noise_proj(noisy_target)

        node_features = x_mamba_out + noise_emb + t_emb

        out_list = []
        for b in range(batch_size):
            graph_out = self.gat(node_features[b], edge_index)
            out_list.append(graph_out)

        final_features = torch.stack(out_list)

        predicted_noise = self.output_proj(final_features) # (Batch, 1937, 20)
        return predicted_noise

print("✅ MarketMamba V3 大腦藍圖繪製完成！容納 20 天軌跡預測能力！")

In [ ]:
# ==========================================
# 🚀 MarketMamba V3.1: Cell 3 - 動態軌跡切片機 (訊號放大對齊版)
# ==========================================
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("🕸️ 啟動 V3.1 動態軌跡切片機 (解決縮放幻覺)...\n")

FEATURE_DIR = '/content/drive/MyDrive/MarketMamba_Database/Macro_1D_Features'
TENSOR_SAVE_PATH = os.path.join(FEATURE_DIR, 'market_tensors_v2_base.pt')

data = torch.load(TENSOR_SAVE_PATH, weights_only=False)
market_matrix = data['market_matrix']
tickers = data['tickers']
num_stocks = market_matrix.shape[1]

# 🌟 宇宙常數：將微小的報酬率放大 10 倍，匹配擴散模型的 N(0,1) 雜訊
SCALE_FACTOR = 10.0

class MarketTrajectoryDataset(Dataset):
    def __init__(self, matrix, seq_len=120, pred_days=30):
        self.matrix = matrix
        self.seq_len = seq_len
        self.pred_days = pred_days
        self.total_days = matrix.shape[0]
        self.valid_indices = list(range(seq_len, self.total_days - pred_days))

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        i = self.valid_indices[idx]
        X = self.matrix[i - self.seq_len : i]

        current_close = self.matrix[i - 1, :, 3]
        future_closes = self.matrix[i : i + self.pred_days, :, 3]

        # 計算原始軌跡
        raw_trajectory = (future_closes - current_close) / (current_close + 1e-8)
        # 🌟 放大訊號！
        scaled_trajectory = raw_trajectory * SCALE_FACTOR

        Y = scaled_trajectory.permute(1, 0)
        return X, Y

SEQ_LEN = 120
PRED_DAYS = 30
BATCH_SIZE = 16  # 保持 16 穩穩跑

dataset = MarketTrajectoryDataset(market_matrix, seq_len=SEQ_LEN, pred_days=PRED_DAYS)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print("\n📊 正在編織 V3.1 GNN 關聯網...")
baseline_closes = market_matrix[-SEQ_LEN:, :, 3].T
corr_matrix = torch.nan_to_num(torch.corrcoef(baseline_closes), 0.0)
corr_matrix.fill_diagonal_(0.0)

TOP_K = 10
edges_source, edges_target = [], []
for i in tqdm(range(num_stocks), desc="編織網路"):
    top_k_indices = torch.topk(corr_matrix[i], TOP_K).indices
    for j in top_k_indices:
        edges_source.append(i)
        edges_target.append(j.item())

edge_index = torch.tensor([edges_source, edges_target], dtype=torch.long)
print(f"🕸️ GNN 關聯網建立完成！訊號已放大 {SCALE_FACTOR} 倍，準備對抗雜訊！\n")

In [ ]:
# ==========================================
# 🚀 MarketMamba V3: Cell 4 - V3 專屬煉丹爐 (支援多步軌跡預測)
# ==========================================
import os
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm.auto import tqdm

print("🔥 啟動 V3 擴容版煉丹爐 (支援 20 天軌跡預測)...")

# ==========================================
# 1. V3 專屬擴散模型排程器 (修正高維度廣播問題)
# ==========================================
class DDPMScheduler:
    def __init__(self, num_timesteps=100, beta_start=0.0001, beta_end=0.02, device="cuda"):
        self.num_timesteps = num_timesteps
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.device = device

    def add_noise(self, original_samples, noise, timesteps):
        """將乾淨的 20 天軌跡 (Y) 加上特定時間步 (t) 的雜訊"""
        alphas_cumprod_t = self.alphas_cumprod[timesteps]
        # ⚠️ 修正：因為 Y 現在是 3D 張量 (Batch, 股票數, 20天)，所以必須用 .view(-1, 1, 1)
        sqrt_alpha_prod = torch.sqrt(alphas_cumprod_t).view(-1, 1, 1)
        sqrt_one_minus_alpha_prod = torch.sqrt(1 - alphas_cumprod_t).view(-1, 1, 1)
        return sqrt_alpha_prod * original_samples + sqrt_one_minus_alpha_prod * noise

# ==========================================
# 2. 初始化 V3 大腦與基礎設施
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ⚠️ 呼叫最新的 MarketMambaV3 擴容版大腦
model = MarketMambaV3(input_dim=10, d_model=128, num_heads=8, pred_days=30).to(device)
scheduler = DDPMScheduler(num_timesteps=100, device=device)

criterion = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# 建立 V3 專屬的儲存資料夾
SAVE_DIR = '/content/drive/MyDrive/MarketMamba_Database/V3_120D_10F'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'MarketMamba_V3_Trajectory.pth')

# ==========================================
# 3. 啟動訓練馬拉松 (含 Early Stopping)
# ==========================================
EPOCHS = 50
best_loss = float('inf')
patience = 5
patience_counter = 0

print("\n🚀 點火！V3 多步預測訓練正式開始...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch_idx, (X_batch, Y_batch) in enumerate(progress_bar):
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
        batch_size = X_batch.shape[0]

        timesteps = torch.randint(0, scheduler.num_timesteps, (batch_size,), device=device).long()
        noise = torch.randn_like(Y_batch)
        y_noisy = scheduler.add_noise(Y_batch, noise, timesteps)

        optimizer.zero_grad()
        predicted_noise = model(X_batch, edge_index.to(device), y_noisy, timesteps)

        loss = criterion(predicted_noise, noise)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_loss = total_loss / len(dataloader)
    print(f"📈 Epoch {epoch+1} 總結 | 平均 Loss: {avg_loss:.5f}")

    if avg_loss < best_loss - 0.0001:
        print(f"🎉 發現更低 Loss ({best_loss:.5f} -> {avg_loss:.5f})！正在備份 V3 大腦...")
        best_loss = avg_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
    else:
        patience_counter += 1
        print(f"⚠️ Loss 未顯著下降，耐心值消耗: {patience_counter} / {patience}")

    if patience_counter >= patience:
        print(f"\n🛑 觸發早停！提早結束訓練！")
        break

print(f"\n💾 V3 大腦已存入: {MODEL_SAVE_PATH}")

In [ ]:
# ==========================================
# 🚀 MarketMamba V3.1: Cell 5 - 時間衰減觀測與資金盤 (現實還原版)
# ==========================================
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

print("📊 啟動 V3.1 Alpha 衰減觀測站與凱利資金盤...\n")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
V3_MODEL_PATH = '/content/drive/MyDrive/MarketMamba_Database/V3_120D_10F/MarketMamba_V3_Trajectory.pth'
PRED_DAYS = 30
SCALE_FACTOR = 10.0 # 🌟 必須與 Dataset 設定的放大倍數一致！

model = MarketMambaV3(input_dim=10, d_model=128, num_heads=8, pred_days=PRED_DAYS)
model.load_state_dict(torch.load(V3_MODEL_PATH, map_location=device, weights_only=True))
model.to(device)
model.eval()

scheduler = DDPMScheduler(num_timesteps=100, device=device)

SEQ_LEN = 120
test_i = market_matrix.shape[0] - PRED_DAYS - 100
test_x = market_matrix[test_i - SEQ_LEN : test_i].clone().detach().unsqueeze(0).to(device)

current_close = market_matrix[test_i - 1, :, 3]
future_closes = market_matrix[test_i : test_i + PRED_DAYS, :, 3]
# 這裡維持原始真實的報酬率
true_trajectory = ((future_closes - current_close) / (current_close + 1e-8)).numpy()

test_closes = test_x[0, :, :, 3].cpu().T
corr_matrix = torch.nan_to_num(torch.corrcoef(test_closes), 0.0)
corr_matrix.fill_diagonal_(0.0)
edges_source, edges_target = [], []
for i in range(len(tickers)):
    for j in torch.topk(corr_matrix[i], 10).indices:
        edges_source.append(i)
        edges_target.append(j.item())
test_edge_index = torch.tensor([edges_source, edges_target], dtype=torch.long).to(device)

@torch.no_grad()
def generate_future_trajectories(model, scheduler, x_latest, edge_idx, num_futures=30):
    num_stocks = x_latest.shape[2]
    x_history_expanded = x_latest.expand(num_futures, -1, -1, -1)
    x_t = torch.randn(num_futures, num_stocks, PRED_DAYS, device=device)

    for i in tqdm(reversed(range(scheduler.num_timesteps)), total=scheduler.num_timesteps, desc="🌌 30天軌跡降噪中"):
        t = torch.full((num_futures,), i, device=device, dtype=torch.long)
        predicted_noise = model(x_history_expanded, edge_idx, x_t, t)
        alpha_t = scheduler.alphas[i]
        alpha_t_cumprod = scheduler.alphas_cumprod[i]
        beta_t = scheduler.betas[i]
        noise = torch.randn_like(x_t) if i > 0 else torch.zeros_like(x_t)
        x_t = (1 / torch.sqrt(alpha_t)) * (x_t - ((1 - alpha_t) / torch.sqrt(1 - alpha_t_cumprod)) * predicted_noise) + torch.sqrt(beta_t) * noise
    return x_t

# 🌟 V3.1 關鍵升級：將模型吐出來的預測值，除以 10 倍，還原成真實台股的百分比尺度！
future_trajectories = generate_future_trajectories(model, scheduler, test_x, test_edge_index, num_futures=30) / SCALE_FACTOR
pred_mean_trajectory = future_trajectories.mean(dim=0).cpu().numpy()

horizons = [5, 10, 15, 20, 25, 30]
print("\n==========================================")
print("⏱️ 【MarketMamba V3.1 時間效力衰減表 (真實尺度)】")
print("==========================================")
print(f"{'預測天數':<10} | {'Rank IC (方向準確度)':<20} | {'MAE (誤差大小)':<15}")
print("-" * 50)
for d in horizons:
    idx = d - 1
    true_y_d = true_trajectory[idx, :]
    pred_y_d = pred_mean_trajectory[:, idx]
    mae = mean_absolute_error(true_y_d, pred_y_d)
    ic_value, _ = spearmanr(pred_y_d, true_y_d)
    print(f"未來 {d:<4} 天 | IC: {ic_value:+.4f} {'🔥' if ic_value > 0.03 else ''}{'🚀' if ic_value > 0.05 else '':<10} | MAE: {mae:.4f}")

KELLY_TARGET_DAY = 20
target_idx = KELLY_TARGET_DAY - 1
target_pred_mean = pred_mean_trajectory[:, target_idx]
target_pred_std = np.clip(future_trajectories[:, :, target_idx].std(dim=0).cpu().numpy(), a_min=1e-6, a_max=None)
sharpe_score = target_pred_mean / target_pred_std

df_eval = pd.DataFrame({
    'Ticker': tickers,
    f'Exp_Return_{KELLY_TARGET_DAY}D': target_pred_mean,
    'Volatility_Risk': target_pred_std,
    'Sharpe_Score': sharpe_score,
})

df_eval['Kelly_Raw'] = np.where(df_eval[f'Exp_Return_{KELLY_TARGET_DAY}D'] > 0, df_eval[f'Exp_Return_{KELLY_TARGET_DAY}D'] / (df_eval['Volatility_Risk'] ** 2), 0)
df_eval['Suggested_Weight'] = np.clip(df_eval['Kelly_Raw'] * 0.5, a_min=0.0, a_max=0.20)

top_picks = df_eval[df_eval['Suggested_Weight'] > 0].sort_values('Sharpe_Score', ascending=False).head(10)
top_picks['Suggested_Weight'] = (top_picks['Suggested_Weight'] * 100).apply(lambda x: f"{x:.2f}%")
top_picks[f'Exp_Return_{KELLY_TARGET_DAY}D'] = (top_picks[f'Exp_Return_{KELLY_TARGET_DAY}D'] * 100).apply(lambda x: f"{x:.2f}%")
top_picks['Volatility_Risk'] = (top_picks['Volatility_Risk'] * 100).apply(lambda x: f"{x:.2f}%")

print("\n==========================================")
print(f"🧑‍💻 【V3.1 現實對齊版 資金決策表 (基於 {KELLY_TARGET_DAY} 天)】")
print("==========================================")
print(top_picks[['Ticker', f'Exp_Return_{KELLY_TARGET_DAY}D', 'Volatility_Risk', 'Sharpe_Score', 'Suggested_Weight']].to_string(index=False))
print("==========================================")

In [ ]:
# ==========================================
# 🔭 MarketMamba V3: Cell 6 - 平行宇宙觀測儀 (30天軌跡切面版)
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("🔭 啟動 MarketMamba V3 平行宇宙觀測儀...\n")

# 🎯 這裡可以自由更改你想觀測的「未來第幾天」 (例如 5, 10, 20, 30)
TARGET_DAY = 20
day_idx = TARGET_DAY - 1

# 你想觀察的股票代號 (建議填入剛剛 V3 評估出來 Top 10 的股票)
target_stocks = ['6856', '6844']

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, target_ticker in enumerate(target_stocks):
    if target_ticker in tickers:
        stock_idx = tickers.index(target_ticker)

        # 🌟 V3 關鍵升級：從 3D 軌跡矩陣中，切出第 TARGET_DAY 天的 30 個平行宇宙預測
        # future_trajectories 形狀: (30宇宙, 1937檔, 30天)
        universe_predictions = future_trajectories[:, stock_idx, day_idx].cpu().numpy() * 100

        # 抓出真實的報酬率
        # true_trajectory 形狀: (30天, 1937檔)
        actual_return = true_trajectory[day_idx, stock_idx] * 100

        mean_pred = universe_predictions.mean()
        std_pred = universe_predictions.std()

        ax = axes[idx]

        # 繪製機率密度雲 (KDE) 與直方圖
        sns.histplot(universe_predictions, kde=True, ax=ax, color="skyblue", bins=10, stat="density", alpha=0.6)

        ax.axvline(mean_pred, color='blue', linestyle='--', linewidth=2, label=f'Predicted Mean: {mean_pred:.2f}%')
        ax.axvline(actual_return, color='red', linestyle='-', linewidth=2, label=f'True Return: {actual_return:.2f}%')

        ax.set_title(f'Stock {target_ticker} - Day {TARGET_DAY} Prediction\n(Std / Risk: {std_pred:.2f}%)', fontsize=14, fontweight='bold')
        ax.set_xlabel(f'Predicted Return at Day {TARGET_DAY} (%)', fontsize=12)
        ax.set_ylabel('Density', fontsize=12)
        ax.legend()

    else:
        print(f"⚠️ 找不到股票 {target_ticker} 的資料。")

plt.tight_layout()
plt.show()

print("==========================================")
print(f"💡 【V3 觀測指南】 目前展示的是未來第 {TARGET_DAY} 天的宇宙切面。")
print("你可以試著把 TARGET_DAY 改成 5 或 30，看看機率鐘形罩會不會隨著時間推移而變寬（不確定性增加）！")
print("==========================================")

In [ ]:
# ==========================================
# 🚀 MarketMamba V3.1: Cell 6 - 網站快取打包中心 (離線前最後準備)
# ==========================================
import os
import pandas as pd
import numpy as np

print("📦 啟動 V3.1 網站快取打包中心...")

# 🌟 1. 改用最精準的「第 15 天」作為資金配置基準
KELLY_TARGET_DAY = 15
target_idx = KELLY_TARGET_DAY - 1

print(f"🎯 鎖定最佳預測區間：未來 {KELLY_TARGET_DAY} 天")

# 提取第 15 天的預測均值與標準差
target_pred_mean = pred_mean_trajectory[:, target_idx]
# 注意：future_trajectories 已經在 Cell 5 除以 SCALE_FACTOR 了
target_pred_std = np.clip(future_trajectories[:, :, target_idx].std(dim=0).cpu().numpy(), a_min=1e-6, a_max=None)
sharpe_score = target_pred_mean / target_pred_std

# 建立完整的評估表
df_export = pd.DataFrame({
    'Ticker': tickers,
    f'Exp_Return_{KELLY_TARGET_DAY}D': target_pred_mean,
    'Volatility_Risk': target_pred_std,
    'Sharpe_Score': sharpe_score,
})

# 計算半凱利與 20% 上限防護
df_export['Kelly_Raw'] = np.where(df_export[f'Exp_Return_{KELLY_TARGET_DAY}D'] > 0,
                                  df_export[f'Exp_Return_{KELLY_TARGET_DAY}D'] / (df_export['Volatility_Risk'] ** 2), 0)
df_export['Suggested_Weight'] = np.clip(df_export['Kelly_Raw'] * 0.5, a_min=0.0, a_max=0.20)

# 排序並濾除不需要買的股票，讓網站讀取更快
df_export = df_export.sort_values('Sharpe_Score', ascending=False)

# 🌟 2. 儲存資金決策表 CSV
SAVE_DIR = '/content/drive/MyDrive/MarketMamba_Database/V3_120D_10F'
EVAL_SAVE_PATH = os.path.join(SAVE_DIR, f'V3_1_Kelly_Allocation_{KELLY_TARGET_DAY}D.csv')
df_export.to_csv(EVAL_SAVE_PATH, index=False)
print(f"✅ 資金決策表已存檔至: {EVAL_SAVE_PATH}")

# 🌟 3. 儲存 30 天連續軌跡 CSV (給網站畫折線圖跟漏斗圖用)
# 把 pred_mean_trajectory (1937, 30) 轉成 DataFrame
df_trajectory = pd.DataFrame(pred_mean_trajectory, columns=[f'Day_{i+1}' for i in range(PRED_DAYS)])
df_trajectory.insert(0, 'Ticker', tickers)

TRAJ_SAVE_PATH = os.path.join(SAVE_DIR, 'V3_1_Mean_Trajectories_30D.csv')
df_trajectory.to_csv(TRAJ_SAVE_PATH, index=False)
print(f"✅ 30 天預測軌跡庫已存檔至: {TRAJ_SAVE_PATH}")

print("\n==========================================")
print("🎉 網站快取打包完成！")
print("你的 Streamlit 網站現在有了最強的資料庫後盾！")
print("==========================================")